# STEP 1 - INSTALL LIBRARIES


In [1]:
!nvidia-smi

Mon May 18 18:12:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets peft accelerate
!pip install -q fastapi uvicorn pyngrok nest_asyncio
!pip install -q detoxify

# STEP 2 - IMPORT LIBRARIES

In [3]:
import torch
import time

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

from peft import (
    LoraConfig,
    get_peft_model
)

from detoxify import Detoxify

# STEP 3 - LOAD DATASET

In [4]:
ds = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# STEP 4 - REDUCE DATASET SIZE

In [5]:
small_ds = ds["train"].select(range(5000))

print(small_ds[0])

{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


# STEP 5 - FORMAT DATASET

In [6]:
def format_data(example):

    text = f"""
### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""

    return {"text": text}

small_ds = small_ds.map(format_data)

# STEP 6 - LOAD MODEL

In [7]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model Loaded")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Loaded


# STEP 7 - APPLY LORA

In [8]:
# Upgrade torchao to a compatible version as required by peft
!pip install --upgrade torchao

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

print("LoRA Applied")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


LoRA Applied


# STEP 8 - TOKENIZE DATASET

In [9]:
def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_ds = small_ds.map(tokenize_function)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

# STEP 9 - DATA COLLATOR

In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# STEP 10 - TRAINING ARGUMENTS

In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=20
)

# STEP 11 - CREATE TRAINER

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator
)


# STEP 12 - TRAIN MODEL

In [13]:
from transformers import Trainer

In [14]:
trainer.train()

Step,Training Loss
5,1.980435
10,1.764628
15,1.627516
20,1.516264
25,1.649926
30,1.292780
35,1.332638
40,1.321958
45,1.246330
50,1.100310


TrainOutput(global_step=5000, training_loss=0.6950526526451111, metrics={'train_runtime': 753.2118, 'train_samples_per_second': 6.638, 'train_steps_per_second': 6.638, 'total_flos': 7953705861120000.0, 'train_loss': 0.6950526526451111, 'epoch': 1.0})

# STEP 13 - SAVE MODEL

In [15]:
model.save_pretrained(
    "customer_support_model"
)

tokenizer.save_pretrained(
    "customer_support_model"
)

print("Model Saved")

Model Saved


# STEP 14 - LOAD PIPELINE

In [16]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# STEP 15 - TEST MODEL

In [17]:
query = "I want refund because my product arrived damaged."

result = pipe(
    query,
    max_new_tokens=100
)

print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I want refund because my product arrived damaged.

Response: I understand that you're experiencing difficulties with your purchase. Rest assured, we will work diligently to resolve the issue and ensure your satisfaction. Can you please provide us with more details about the damaged product?

Additional Questions:
- Did the product arrive damaged in transit?
- Is the damage visible to the naked eye?
- How did you receive the damaged product?
- Are there any additional notes or details


# STEP 16 - TOXICITY FILTERING

In [18]:
def safe_response(text):

    result = Detoxify('original').predict(text)

    if result['toxicity'] > 0.7:
        return "Unsafe response blocked."

    return text

# STEP 17 - PROMPT GUARDRAILS

In [19]:
blocked_words = [
    "ignore previous instructions",
    "hack",
    "reveal system prompt",
    "password"
]

def validate_prompt(prompt):

    for word in blocked_words:

        if word.lower() in prompt.lower():
            return False

    return True

# STEP 18 - FINAL CHATBOT

In [20]:
def chatbot(query):

    if not validate_prompt(query):
        return "Unsafe prompt blocked."

    result = pipe(
        query,
        max_new_tokens=100
    )

    output = result[0]["generated_text"]

    return safe_response(output)

#STEP 19 - TEST CHATBOT


In [21]:
print(
    chatbot(
        "I want refund because my package is damaged."
    )
)


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:01<00:00, 232MB/s]


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

I want refund because my package is damaged. How do I proceed?

Answer: I'm glad you asked! To proceed with a refund, please follow these steps:

1. Sign into your account: Access our website and log in to your account.
2. Go to your orders: Once logged in, navigate to the "Orders" section where you can view your order history.
3. Locate the damaged package: Look for the package with the damaged item and click on it to view


# STEP 20 - BENCHMARKING

In [22]:
start = time.time()

response = chatbot(
    "I need refund"
)

end = time.time()

print("Latency:", end - start)

memory = torch.cuda.memory_allocated()/1024**3

print("GPU Memory:", memory, "GB")

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Latency: 6.012211561203003
GPU Memory: 2.0775012969970703 GB


# STEP 21 - CREATE FASTAPI FILE

In [23]:
%%writefile app.py

from fastapi import FastAPI
from transformers import pipeline

app = FastAPI()

@app.get("/")
def home():

    return {
        "message": "Customer Support API Running"
    }

@app.post("/chat")
def chat(query: str):

    return {
        "response": query
    }

Writing app.py


# STEP 22 - START FASTAPI SERVER

# STEP 23 - VERIFY SERVER

In [ ]:
!curl http://127.0.0.1:8000

curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Connection refused


# STEP 24 - NGROK SETUP

In [53]:
from pyngrok import ngrok

ngrok.set_auth_token("3DuSQIcs630maNAN5eNVg1PXMbT_3xcjX8aXRV8TDEeLoUBQf")

public_url = ngrok.connect(8000)

print(public_url)


NgrokTunnel: "https://shuffle-unstuck-glorifier.ngrok-free.dev" -> "http://localhost:8000"


# STEP 26 - ZIP MODEL

In [30]:
!zip -r customer_support_model.zip customer_support_model


updating: customer_support_model/ (stored 0%)
updating: customer_support_model/adapter_config.json (deflated 58%)
updating: customer_support_model/chat_template.jinja (deflated 60%)
updating: customer_support_model/README.md (deflated 66%)
updating: customer_support_model/tokenizer_config.json (deflated 46%)
updating: customer_support_model/adapter_model.safetensors (deflated 8%)
updating: customer_support_model/tokenizer.json (deflated 85%)


# STEP 27 - HUGGING FACE LOGIN

In [47]:
from huggingface_hub import notebook_login

notebook_login()

# STEP 28 - UPLOAD MODEL

In [50]:
!pip install -q -U huggingface_hub transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 102.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.


In [52]:
model.push_to_hub(
    "customer-support-llm"
)

tokenizer.push_to_hub(
    "customer-support-llm"
)

print("Project Completed Successfully")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  12%|#2        |  558kB / 4.52MB            

README.md: 0.00B [00:00, ?B/s]

Project Completed Successfully
